# Parse HTML Docs
This notebook contains functions that:
1) Retrieves a list of fanwork links from a text file created with the functions in get_fanwork_links.ipynb
2) For each fanwork link, creates a BeautifulSoup object, parses the resulting html document, and extracts desired metadata about each fanwork
3) Saves the extracted metadata in a json structure

In [164]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## TODO - remove once you've tested selenium
import requests

from bs4 import BeautifulSoup
import re
import json

# Retrieve fanworks list from text file

In [165]:
def get_fanwork_links(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    fanwork_links = open(filepath).readlines()
    for i in range(len(fanwork_links)):
        fanwork_links[i] = fanwork_links[i].replace("\n","")
    return fanwork_links

# Create soup objects

In [166]:
def get_fanwork_html(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()

    else:
        pass

    # detect adult content warning
    accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
    print("Adult content warning is present: ", str(accept_adult_content_present))

    if accept_adult_content_present:
        wait = WebDriverWait(driver, 15)
        accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

        # click link + accept adult content warning
        actions = ActionChains(driver)
        actions.move_to_element(accept_adult_content_button).perform()
        actions.click(accept_adult_content_button).perform()

    else:
        pass

    # return page source / html doc
    driver.implicitly_wait(10) # seconds

    # get OuterHTML from 'work meta group'
    work_meta_group = driver.find_element(By.CLASS_NAME, 'wrapper')
    outer_html = work_meta_group.get_attribute('outerHTML')

    return outer_html

In [416]:
## TODO - delete once you've tested the other one
def get_html_for_date(url):
    """Takes in the url for any given Ao3 fanwork and returns an html document of that page."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [167]:
def get_soup(html):
    soup = BeautifulSoup(html, features='html.parser')
    return soup

In [676]:
def detect_adult_content_warning(soup):
    h2 = soup.find('h2')
    string = str(h2.string)
    if string == "Adult Content Warning":
        adult_content_present = True
    else:
        adult_content_present = False
    return adult_content_present

In [168]:
def get_all_dt(soup):
    """Takes in BeautifulSoup object from Ao3 fanwork and returns a list of all 'dt' elements of the html document. 'dt' elements represent the metadata categories present for the fanwork."""
    all_dt = soup.find_all('dt')
    # all_dt = list(all_dt)
    return all_dt

In [169]:
def update_all_dt(dt_list):
    dt_dict = {
        '<dt><label for="user_session_login_small">Username or email:</label></dt>':'user_session_login',
        '<dt><label for="user_session_password_small">Password:</label></dt>':'user_session_password',
        '<dt class="rating tags">':'Rating',
        '<dt class="warning tags">':'Archive Warning',
        '<dt class="category tags">':'Categories',
        '<dt class="fandom tags">':'Fandoms',
        '<dt class="relationship tags">':'Relationships',
        '<dt class="character tags">':'Characters',
        '<dt class="freeform tags">':'Additional Tags',
        '<dt class="language">':'Language',
        '<dt class="stats">':'Stats',
        '<dt class="published">':'Date Published',
        '<dt class="status">':'Date Updated',
        '<dt class="words">':'Word Count',
        '<dt class="chapters">':'Chapters',
        '<dt class="comments">':'Comments',
        '<dt class="kudos">':'Kudos',
        '<dt class="bookmarks">':'Bookmarks',
        '<dt class="hits">':'Hits',
        '<dt class="landmark">Note:</dt>':'note',
        '<dt><label for="comment_name_for_211688701">Guest name</label></dt>':'guest_commenter_name',
        '<dt><label for="comment_email_for_211688701">Guest email</label></dt>':'guest_commenter_email'
    }

    for i in range(len(dt_list)):
        dt_list[i] = str(dt_list[i])

    for key in dt_dict.keys():
        for i in range(len(dt_list)):
            key_string = str(key)
            if key_string in dt_list[i]:
                dt_list[i] = dt_dict[key]
            else:
                continue
    return dt_list

In [170]:
def get_all_dd(soup):
    """Takes in BeautifulSoup object from Aoe fanwork and returns a list of all 'dd' elements of the html document. 'dd' elements represent the metadata values present in the fanwork."""
    all_dd = soup.find_all('dd')
    # all_dd = list(all_dd)

    # convert dd_list items to strings
    for i in range(len(all_dd)):
        all_dd[i] = str(all_dd[i])

    return all_dd

# Parse html docs and clean metadata

## strip_dd

In [171]:
def strip_dd(dd_list):
    text_to_strip = ['\n',
                 '<ul class="commas">',
                 '<li>','</li>',
                 '</ul>',
                 '<a class="tag" href="','</a>',
                 '<dd class="rating tags">',
                 '/tags/Not%20Rated/works">',
                 '/tags/General%20Audiences/works">',
                 '/tags/Teen%20And%20Up%20Audiences/works">',
                 '/tags/Mature/works">',
                 '/tags/Explicit/works">',
                 '<dd class="warning tags">',
                 '/tags/No%20Archive%20Warnings%20Apply/works">',
                 '/tags/Graphic%20Depictions%20Of%20Violence/works">',
                 '/tags/Choose%20Not%20To%20Use%20Archive%20Warnings/works">',
                 '/tags/Major%20Character%20Death/works">',
                 '/tags/Rape*s*Non-Con/works">',
                 '/tags/Underage%20Sex/works">',
                 '<dd class="category tags">',
                 '/tags/Gen/works">',
                 '/tags/F*s*F/works">',
                 '/tags/F*s*M/works">F/M',
                 '/tags/M*s*M/works">M/M',
                 '/tags/Multi/works">Multi',
                 '/tags/Other/works">Other',
                 '</dd>',
                 '<dd class="language" lang="en">',
                 '<!-- end of cache -->',
                 '<dd class="stats">',
                 '<dl class="stats">',
                 '<dt class="published">',
                 '</dt><dd class="published">',
                 '<dt class="status">',
                 '</dt><dd class="status">',
                 '<dt class="words">',
                 '</dt><dd class="words">',
                 '<dt class="chapters">',
                 '</dt><dd class="chapters">',
                 '<dt class="comments">',
                 '</dt><dd class="comments">',
                 '<dt class="kudos">',
                 '</dt><dd class="kudos">',
                 '<dt class="bookmarks">',
                 '</dt><dd class="bookmarks">',
                 'bookmarks">',
                 '<a href="/',
                 '<dt class="hits">',
                 '</dt><dd class="hits">',
                 '</dl>',
                 '<dd class="instructions comment_form">',
                 '<dd><input id="',
                 '" name="comment[name]" type="text"/><script>',
                 '/<![CDATA[var ',
                 ' = new LiveValidation',
                 ', { wait: 500, onlyOnBlur: false }',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your name.","validMessage":""})',
                 ';//]]>',
                 '</script>',
                 '<input id="',
                 '" name="comment[email]"',
                 ' type="text"/>',
                 '<script>',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your email address.","validMessage":""})',
                 '<dd><input autocomplete="on" ',
                 'id="user_session_login_small" '
                 ]
    for item in text_to_strip:
        for i in range(len(dd_list)):
            if item in dd_list[i]:
                dd_list[i] = dd_list[i].replace(item, "")
            else:
                continue
    return dd_list

# Retrieve Metadata

In [687]:
def get_rating_adult(soup):
    rating_class = soup.find_all(attrs={'class':'rating'})
    if len(rating_class) == 0:
        rating_list = "Rating not tagged"
    else:
        rating_list = []
        for item in rating_class:
            tag = item
            string = tag.string
            rating_list.append(string)
        rating_list = list(set(rating_list))
        for i in range(len(rating_list)):
            rating_list[i] = str(rating_list[i])
        if len(rating_list) == 1:
            rating_list = rating_list[0]
        else:
            pass
    return rating_list

In [681]:
def get_rating_nonadult(soup):
    """Get rating for fanworks that do NOT have the adult content warning."""
    rating_class = soup.find_all(attrs={'class':'rating tags'})
    if len(rating_class) == 0:
        rating = 'Unknown'
    else:
        rating_class1 = rating_class[1]
        tag = rating_class1.find('a')
        string = str(tag.string)
        rating = string
    return rating

In [589]:
## TODO - delete after saving next draft
def get_rating1(soup):
    rating_class = soup.find_all(attrs={'class':'rating'})
    if rating_class is None:
        rating_class = soup.find_all(attrs={'class':'rating tags'})
        if len(rating_class) == 0:
            rating = "Unknown"
            return rating
        else:
            rating_class1 = rating_class[1]
            tag = rating_class1.find('a')
            string = str(tag.string)
            rating = string
            return rating

    else:
        for child in rating_class:
            tag = child.find('a')
            if tag is not None:
                string = str(tag.string)
                rating = string
                return rating

In [693]:
def get_warnings_adult(soup):
    warning_class = soup.find_all(attrs={'class':'warnings'})
    if len(warning_class) == 0:
        warning_list = "Warnings not tagged"
    else:
        warnings_list = []
        for item in warning_class:
            tag = item
            string = tag.string
            warnings_list.append(string)
        warnings_list = list(set(warnings_list))
        for i in range(len(warnings_list)):
            warnings_list[i] = str(warnings_list[i])
        if len(warnings_list) == 1:
            warnings_list = warnings_list[0]
        else:
            pass
    return warnings_list

In [696]:
def get_warnings_nonadult(soup):
    warnings_list = []
    warning_class = soup.find_all(attrs={'class':'warning tags'})
    if len(warning_class) == 0:
        warnings_list = "Warnings not tagged"
    else:
        warning_class1 = warning_class[1]
        tag = warning_class1.find('a')
        string = str(tag.string)
        warnings_list.append(string)
        if len(warnings_list) == 1:
            warnings_list = warnings_list[0]
    return warnings_list

In [625]:
## TODO - delete after saving next draft
def get_warnings1(soup):
    warning_class = soup.find_all(attrs={'class':'warning'})
    if warning_class is None:
        warnings_list = []
        warning_class = soup.find_all(attrs={'class':'warning tags'})
        if len(warning_class) == 0:
            warnings_list = "Warnings not tagged"
            return warnings_list
        else:
            warning_class1 = warning_class[1]
            tag = warning_class1.find('a')
            string = str(tag.string)
            warnings_list.append(string)
            return warnings_list
    else:
        warning_class1 = warning_class[1]
        warnings_list = []
        for child in warning_class1:
            tag = warning_class1.find('a')
            if tag is not None:
                string = str(tag.string)
                warnings_list.append(string)
            else:
                pass
        warnings_list = list(set(warnings_list))
        return warnings_list

In [697]:
def get_category_adult(soup):
    category_class = soup.find_all(attrs={'class':'category'})
    if len(category_class) == 0:
        category_list = "No categories tagged"
    else:
        category_list = []
        for item in category_class:
            tag = item
            string = tag.string
            category_list.append(string)
        rating_list = list(set(category_list))
        for i in range(len(category_list)):
            category_list[i] = str(category_list[i])
        if len(category_list) == 1:
            category_list = category_list[0]
        else:
            pass
    return category_list

In [698]:
def get_category_nonadult(soup):
    category_class = soup.find_all(attrs={'class':'category tags'})
    if len(category_class) == 0:
        category = "No categories tagged"
    else:
        category_class1 = category_class[1]
        tag = category_class1.find('a')
        string = str(tag.string)
        category = string
    return category

In [502]:
## TODO - delete when you save next draft
def get_category1(soup):
    category_class = soup.find_all(attrs={'class':'category'})
    if len(category_class) == 0:
        category_class = soup.find_all(attrs={'class':'category tags'})
        if len(category_class) == 0:
            category = "No categories tagged"
            return category
        else:
            category_class1 = category_class[1]
            tag = category_class1.find('a')
            string = str(tag.string)
            category = string
            return category
    else:
        category_list = []
        for item in category_class:
            tag = item
            string = tag.string
            category_list.append(string)
        rating_list = list(set(category_list))
        for i in range(len(category_list)):
            category_list[i] = str(category_list[i])
        if len(category_list) == 1:
            category_list = category_list[0]
        else:
            pass
    return category_list

In [703]:
def get_language_adult(soup):
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language = language_list[1]
    return (language)

In [704]:
def get_language_nonadult(soup):
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language = language_list[1]
    return (language)

In [712]:
def get_fandom_adult(soup):
    fandom_class = soup.find(attrs={'class':'fandoms heading'}).find('a')
    fandom_list = []
    for item in fandom_class:
        string = str(item.string)
        fandom_list.append(string)
    fandom_list = list(set(fandom_list))
    if len(fandom_list) == 1:
        fandom_list = fandom_list[0]
    else:
        pass
    return fandom_list

In [713]:
def get_fandom_nonadult(soup):
    fandom_class = soup.find_all(attrs={'class':'fandom tags'})
    fandom_class1 = fandom_class[1]
    fandom_list = []
    for child in fandom_class1:
        tag = fandom_class1.find('a')
        string = tag.string
        fandom_list.append(string)
    fandom_list = list(set(fandom_list))
    if len(fandom_list) == 1:
        fandom = fandom_list[0]
        return fandom
    else:
        return fandom_list

In [530]:
# TODO - delete when you save the next draft
def get_fandom1(soup):
    fandom_class = soup.find(attrs={'class':'fandoms heading'})
    if fandom_class is None:
        fandom_class = soup.find_all(attrs={'class':'fandom tags'})
        fandom_class1 = fandom_class[1]
        fandom_list = []
        for child in fandom_class1:
            tag = fandom_class1.find('a')
            string = tag.string
            fandom_list.append(string)
        fandom_list = list(set(fandom_list))
        if len(fandom_list) == 1:
            fandom = fandom_list[0]
            return fandom
        else:
            return fandom_list

    else:
        fandom_class_links = soup.find(attrs={'class':'fandoms heading'}).find('a')
        fandom_list = []
        for item in fandom_class_links:
            string = str(item.string)
            fandom_list.append(string)
        fandom_list = list(set(fandom_list))
        if len(fandom_list) == 1:
            fandom_list = fandom_list[0]
        else:
            pass
        return fandom_list

In [716]:
def get_relationships_adult(soup):
    relationships_class = soup.find_all(attrs={'class':'relationships'})
    if len(relationships_class) == 0:
        relationships_list = 'No relationships tagged'
    else:
        relationships_list = []
        for item in relationships_class:
            string = str(item.string)
            relationships_list.append(string)
        if len(relationships_list) == 1:
            relationships_list = relationships_list[0]
        else:
            pass
    return relationships_list

In [757]:
def get_relationships_nonadult(soup):
    relationships_class = soup.find_all(attrs={'class':'relationship tags'})
    if len(relationships_class) == 0:
        relationships_list = "No relationships tagged"
    else:
        relationships_class1 = relationships_class[1]
        for child in relationships_class1:
            tag_list = relationships_class1.find_all('a')
            relationships_list = []
            for tag in tag_list:
                string = str(tag.string)
                relationships_list.append(string)
            if len(relationships_list) == 1:
                relationships_list = relationships_list[0]
            else:
                pass
    return relationships_list

In [536]:
# TODO - delete when you save next draft
def get_relationships1(soup):
    relationships_class = soup.find_all(attrs={'class':'relationships'})
    if relationships_class is None:
        relationships_class = soup.find_all(attrs={'class':'relationship tags'})
        if len(relationships_class) == 0:
            relationships_list = "No relationships tagged"
        else:
            relationships_class1 = relationships_class[1]
            for child in relationships_class1:
                tag_list = relationships_class1.find_all('a')
                relationships_list = []
                for tag in tag_list:
                    string = str(tag.string)
                    relationship_list.append(string)
    else:
        relationships_list = []
        for item in relationships_class:
            string = str(item.string)
            relationships_list.append(string)

    if len(relationships_list) == 1:
        relationships_list = relationships_list[0]
    else:
        relationships_list = relationships_list
    return relationships_list

In [759]:
def get_characters_adult(soup):
    character_class = soup.find_all(attrs={'class':'characters'})
    if len(character_class) == 0:
        character_list = 'No characters tagged'
    else:
        character_list = []
        for item in character_class:
            string = str(item.string)
            character_list.append(string)
    return character_list

In [760]:
def get_characters_nonadult(soup):
    character_class = soup.find_all(attrs={'class':'character tags'})
    if len(character_class) == 0:
        character_list = "No characters tagged"
    else:
        character_class1 = character_class[1]
        for child in character_class1:
            tag_list = character_class1.find_all('a')
            character_list = []
            for tag in tag_list:
                string = str(tag.string)
                character_list.append(string)
    return character_list

In [537]:
# TODO - delete when you save next draft
def get_characters1(soup):
    character_class = soup.find_all(attrs={'class':'characters'})
    if character_class is None:


    else:
        character_list = []
        for item in character_class:
            string = str(item.string)
            character_list.append(string)

    if len(character_list) == 1:
        character_list = character_list[0]
    else:
        pass

    return character_list

In [761]:
def get_additional_adult(soup):
    freeform_class = soup.find_all(attrs={'class':'freeforms'})
    if len(freeform_class) == 0:
        freeform_list = 'No additional tags'
    else:
        freeform_list = []
        for item in freeform_class:
            string = str(item.string)
            freeform_list.append(string)
    return freeform_list

In [762]:
def get_additional_nonadult(soup):
    freeform_class = soup.find_all(attrs={'class':'freeform tags'})
    if len(freeform_class) == 0:
        freeform_list = "No additional tags"
    else:
        freeform_class1 = freeform_class[1]
        for child in freeform_class1:
            tag_list = freeform_class1.find_all('a')
            freeform_list = []
            for tag in tag_list:
                string = str(tag.string)
                freeform_list.append(string)
    return freeform_list

In [538]:
# TODO - delete when save next draft
def get_additional_tags1(soup):
    freeform_class = soup.find_all(attrs={'class':'freeforms'})
    if freeform_class is None:
        freeform_class = soup.find_all(attrs={'class':'freeform tags'})
        if len(freeform_class) == 0:
            freeform_list = "No additional tags"
        else:
            freeform_class1 = freeform_class[1]
            for child in freeform_class1:
                tag_list = freeform_class1.find_all('a')
                freeform_list = []
                for tag in tag_list:
                    string = str(tag.string)
                    freeform_list.append(string)
    else:
        freeform_list = []
        for item in freeform_class:
            string = str(item.string)
            freeform_list.append(string)

    if len(freeform_list) == 1:
        freeform_list = freeform_list[0]
    else:
        pass

    return freeform_list

In [184]:
def is_series(dt_list):
    for i in range(len(dt_list)):
        current_string = dt_list[i]
        if 'Series' in current_string:
            is_part_of_series = True
            return is_part_of_series
        else:
            is_part_of_series = False
            return is_part_of_series

In [763]:
def get_date_published_adult(soup):
    published_class = soup.find_all(attrs={'class':'published'})
    if len(published_class) == 0:
        published_list = 'Unknown'
    else:
        published_list = []
        for item in published_class:
            string = str(item.string)
            published_list.append(string)
    return published_list

In [185]:
def get_date_published(dt_list, dd_list):
    # find date published index
    if 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_published = dd_list[index]
        date_published = date_published.split('>')
        date_published = date_published[1]
        return date_published

    else:
        date_published = 'Unknown'
        return date_published

In [824]:
def get_date_published_nonadult(soup):
    published_class = soup.find_all(attrs={'class':'published'})
    if len(published_class) == 0:
        published_list = 'Unknown'
    else:
        published_list = []
        for item in published_class:
            string = str(item.string)
            published_list.append(string)
        published_list = published_list[1]
    return published_list

In [825]:
def get_date_updated_nonadult(soup):
    updated_class = soup.find_all(attrs={'class':'status'})
    if len(updated_class) == 0:
        updated_list = 'Unknown'
    else:
        updated_list = []
        for item in updated_class:
            string = str(item.string)
            updated_list.append(string)
        updated_list = updated_list[1]
    return updated_list

In [186]:
def get_date_updated(dt_list, dd_list):
    # find date updated index
    if 'Date Updated' in dt_list:
        index = dt_list.index("Date Updated")
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    elif 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    else:
        date_updated = 'Unknown'
        return date_updated

In [829]:
def get_date_updated_adult(soup):
    updated_class = soup.find_all(attrs={'class':'datetime'})
    if len(updated_class) == 0:
        date_updated = 'Unknown'
    else:
        updated_list = []
        for item in updated_class:
            string = str(item.string)
            updated_list.append(string)
        date_updated = updated_list[0]
    return date_updated

In [807]:
def get_word_count(soup):
    word_count_class = soup.find_all(attrs={'class':'words'})
    word_count_list = []
    for item in word_count_class:
        string = str(item.string)
        word_count_list.append(string)
    word_count = word_count_list[1]
    word_count = word_count.replace(",","")
    word_count = int(word_count)
    return word_count

In [808]:
def get_chapter_count(soup):
    chapter_count_class = soup.find_all(attrs={'class':'chapters'})
    chapter_count_list = []
    for item in chapter_count_class:
        string = str(item.string)
        chapter_count_list.append(string)
    chapter_count = chapter_count_list[1]
    return chapter_count

In [776]:
def get_comment_count_adult(soup):
    comments_class = soup.find_all(attrs={'class':'comments'})
    if len(comments_class) == 0:
        comment_count = 0
    else:
        comment_count_list = []
        for item in comments_class:
            string = str(item.string)
            comment_count_list.append(string)
        comment_count = comment_count_list[1]
    return comment_count

In [805]:
def get_comment_count_nonadult(soup):
    comments_class = soup.find_all(attrs={'class':'comments'})
    string = str(comments_class[2].string)
    comments_count = int(string)
    return comments_count

In [809]:
def get_kudos_count(soup):
    kudos_class = soup.find_all(attrs={'class':'kudos'})
    if len(kudos_class) == 0:
        kudos_count = 0
    else:
        kudos_count_list = []
        for item in kudos_class:
            string = str(item.string)
            kudos_count_list.append(string)
        kudos_count = kudos_count_list[1]
    return kudos_count

In [780]:
def get_bookmarks_count_adult(soup):
    bookmarks_class = soup.find_all(attrs={'class':'bookmarks'})
    if len(bookmarks_class) == 0:
        bookmarks_count = 0
    else:
        bookmarks_count_list = []
        for item in bookmarks_class:
            string = str(item.string)
            bookmarks_count_list.append(string)
        bookmarks_count = bookmarks_count_list[1]
    return bookmarks_count

In [840]:
def get_hits_count_adult(soup):
    hits_class = soup.find_all(attrs={'class':'hits'})
    if len(hits_class) == 0:
        hits_count = 0
    else:
        hits_count_list = []
        for item in hits_class:
            string = str(item.string)
            hits_count_list.append(string)
        hits_count = hits_count_list[1]
    return hits_count

In [841]:
get_hits_count_adult(soup1)

'29'

In [837]:
def get_bookmarks_count_nonadult(soup):
    bookmarks_class = soup.find_all(attrs={'class':'bookmarks'})
    string = str(bookmarks_class[1].string)
    bookmarks_count = int(string)
    return bookmarks_count

In [838]:
get_bookmarks_count_nonadult(soup2)

8

In [842]:
def get_hits_count_nonadult(soup):
    hits_class = soup.find_all(attrs={'class':'hits'})
    string = str(hits_class[1].string)
    hits_count = int(string)
    return hits_count

In [843]:
get_hits_count_nonadult(soup2)

270

# Create Dictionary from html items

In [418]:
def create_fanwork_dict(soup):
    fanwork_dict = {}

    # add rating
    fanwork_dict['Rating'] = get_rating1(soup)

    # add archive warning(s)
    fanwork_dict['Archive Warnings'] = get_warnings1(soup)

    # category
    fanwork_dict['Category'] = get_category1(soup)

    # add fandom
    fanwork_dict['Fandom'] = get_fandom1(soup)

    # add characters
    fanwork_dict['Characters'] = get_characters1(soup)

    # add relationships
    fanwork_dict['Relationships'] = get_relationships1(soup)

    # add additional tags
    fanwork_dict['Additional Tags'] = get_additional_tags1(soup)

    # add language
    fanwork_dict['Language'] = get_language1(soup)

    # add whether it is a series
    # fanwork_dict['Is Series'] = is_series(dd_list)

    # add stats
    fanwork_dict['Date Published'] = get_date_published1(soup)
    fanwork_dict['Date Updated'] = get_date_updated1(soup)
    fanwork_dict['Word Count'] = get_word_count1(soup)
    fanwork_dict['Chapter Count'] = get_chapter_count1(soup)
    fanwork_dict['Comment Count'] = get_comment_count1(soup)
    fanwork_dict['Kudos Count'] = get_kudos_count1(soup)
    fanwork_dict['Bookmarks Count'] = get_bookmarks_count1(soup)

    return fanwork_dict

# Add fanworks_dict to master dictionary (all_works_dict)

In [193]:
def create_all_works_dict():
    dict_all_works = {}
    return dict_all_works

In [194]:
def append_all_works_dict(dict_fanwork, dict_all_works):
    current_length = len(dict_all_works)
    new_idx = current_length + 1
    dict_all_works[new_idx] = dict_fanwork
    return dict_all_works

# Convert master dictionary to json object

In [195]:
def convert_to_json(dict_all_works):
    json_string = json.dumps(dict_all_works, indent=4)
    return json_string

In [424]:
def save_json_to_file(filepath, json_string):
    with open(filepath, "w") as f:
        f.write(json_string)

# Testing

In [500]:
# test get_fanwork_links
fanwork_links = get_fanwork_links('../../data/fanwork_links.txt')

In [429]:
print(len(fanwork_links))

211


## Individual link

In [ ]:
# GOD ok I feel like I need two separate functions for if a work has adult content or not, shit
# I fucking hate this so much

### Test link 1 - has adult content

In [683]:
# test get_html_doc_fanworks
test_url = 'https://archiveofourown.org/works/73925776'
outer_html = get_fanwork_html(test_url)
print(outer_html)
soup1 = get_soup(outer_html)
print(soup1)
# one_fanwork_dict = create_fanwork_dict(soup)
# print(one_fanwork_dict)
# all_works_one = create_all_works_dict()
# append_all_works_dict(one_fanwork_dict, all_works_one)
# one_fanwork_json = convert_to_json(all_works_one)
# filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/one_fanwork_test.json'
# save_json_to_file(filepath, one_fanwork_json)

TOS agreement section is present:  True
Adult content warning is present:  True
<div id="outer" class="wrapper">
      <ul id="skiplinks"><li><a href="#main">Main Content</a></li></ul>
      <noscript><p id="javascript-warning">While we&#39;ve done our best to make the core functionality of this site accessible without JavaScript, it will work better with it enabled. Please consider turning it on!</p></noscript>

<!-- BEGIN header -->

<header id="header" class="region">

  <h1 class="heading">
    <a href="/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"></a>
  </h1>

    <div id="login" class="dropdown" aria-haspopup="true">
      <p class="user actions">
        <a id="login-dropdown" href="/users/login?return_to=%2Fworks%2F73925776" class="dropdown-toggle" data-toggle="dropdown" data-target="#">Log In</a>
      </p>
      <div id="small_login" class="simple login">
	<form class="new_user" id="new_user_session_small" ac

In [705]:
test_get_rating_adult = get_rating_adult(soup1)
print(test_get_rating_adult)

Explicit


In [700]:
test_get_warnings_adult = get_warnings_adult(soup1)
print(test_get_warnings_adult)

No Archive Warnings Apply


In [699]:
test_get_category_adult = get_category_adult(soup1)
print(test_get_category_adult)

M/M


In [706]:
get_language_adult(soup1)

'English'

In [707]:
get_language_nonadult(soup1)

'English'

In [833]:
get_language_adult(soup1)

'English'

In [714]:
get_fandom_adult(soup1)

'Candela Obscura (Critical Role Web Series)'

In [718]:
get_relationships_adult(soup1)

'Leo Amicus/Original Character(s)'

In [764]:
get_characters_adult(soup1)

['Leo Amicus', 'Original Male Character(s)']

In [766]:
get_additional_adult(soup1)

['Circle of the Crimson Mirror (Candela Obscura)',
 'Boats and Ships',
 'Crew of the Dandridge',
 'Missing Scene',
 'Blow Jobs',
 'Anal Sex',
 'Topping from the Bottom',
 'Plot What Plot/Porn Without Plot',
 'almost',
 'Age Difference']

In [769]:
get_date_published_adult(soup1)

'Unknown'

In [834]:
get_date_updated_adult(soup1)

'09 Nov 2025'

In [782]:
get_comment_count_adult(soup1)

'2'

In [811]:
get_kudos_count(soup1)

'2'

In [810]:
get_word_count(soup1)

1070

In [781]:
get_bookmarks_count_adult(soup1)

0

In [812]:
get_chapter_count(soup1)

'1/2'

### test link 2 - does not have adult content

In [617]:
test_url1 = fanwork_links[1]
print(test_url1)

https://archiveofourown.org/works/51518734


In [678]:
# test get_html_doc_fanworks
test_url = fanwork_links[1]
outer_html = get_fanwork_html(test_url)
print(outer_html)
soup2 = get_soup(outer_html)
# print(soup)
# one_fanwork_dict = create_fanwork_dict(soup)
# all_works_one = create_all_works_dict()
# append_all_works_dict(one_fanwork_dict, all_works_one)
# one_fanwork_json = convert_to_json(all_works_one)
# filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/one_fanwork_test1.json'
# save_json_to_file(filepath, one_fanwork_json)

TOS agreement section is present:  True
Adult content warning is present:  False
<div id="outer" class="wrapper">
      <ul id="skiplinks"><li><a href="#main">Main Content</a></li></ul>
      <noscript><p id="javascript-warning">While we&#39;ve done our best to make the core functionality of this site accessible without JavaScript, it will work better with it enabled. Please consider turning it on!</p></noscript>

<!-- BEGIN header -->

<header id="header" class="region">

  <h1 class="heading">
    <a href="/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"></a>
  </h1>

    <div id="login" class="dropdown" aria-haspopup="true">
      <p class="user actions">
        <a id="login-dropdown" href="/users/login?return_to=%2Fworks%2F51518734%2Fchapters%2F130205587" class="dropdown-toggle" data-toggle="dropdown" data-target="#">Log In</a>
      </p>
      <div id="small_login" class="simple login">
	<form class="new_user" id="ne

In [690]:
test_get_rating_nonadult = get_rating_nonadult(soup2)
print(test_get_rating_nonadult)

Teen And Up Audiences


In [701]:
test_get_warnings_nonadult = get_warnings_nonadult(soup2)
print(test_get_warnings_nonadult)

Graphic Depictions Of Violence


In [702]:
test_get_category_nonadult = get_category_nonadult(soup2)
print(test_get_category_nonadult)

F/M


In [767]:
get_language_nonadult(soup2)

'\n        English\n      '

In [710]:
get_language_adult(soup2)

'\n        English\n      '

In [813]:
get_language_nonadult(soup2)

'\n        English\n      '

In [715]:
get_fandom_nonadult(soup2)

'Candela Obscura (Critical Role Web Series)'

In [758]:
get_relationships_nonadult(soup2)

'Jinnah "Jean" Basar/Marion Collodi'

In [765]:
get_characters_nonadult(soup2)

['Jinnah "Jean" Basar', 'Marion Collodi', 'Beatrix Monroe | Auntie Bee']

In [768]:
get_additional_nonadult(soup2)

["Sean is also very present but he's not like. conscious",
 "there's ot3 energy here but this fic is About marion and jean",
 'Pre-Canon',
 'Surgery',
 'Cuddles',
 'get loved idiot']

In [814]:
get_word_count(soup2)

2779

In [815]:
get_kudos_count(soup2)

'52'

In [816]:
get_chapter_count(soup2)

'2/2'

In [806]:
get_comment_count_nonadult(soup2)

15

In [823]:
get_date_published_nonadult(soup2)

'2023-11-11'

In [826]:
get_date_updated_nonadult(soup2)

'2026-04-26'

In [667]:
string = str(all_h2.string)
print(string)


    carry me home on your shoulder
  


In [640]:
print(one_fanwork_dict)

{'Rating': 'Teen And Up Audiences', 'Archive Warnings': ['Graphic Depictions Of Violence'], 'Category': 'F/M', 'Fandom': 'Candela Obscura (Critical Role Web Series)', 'Characters': [], 'Relationships': [], 'Additional Tags': [], 'Language': '\n        English\n      ', 'Date Published': ['Published:', '2023-11-11'], 'Date Updated': 'Unknown', 'Word Count': 2779, 'Chapter Count': '2/2', 'Comment Count': 'Comments:', 'Kudos Count': '52', 'Bookmarks Count': '8'}


In [630]:
category_class = soup.find_all(attrs={'class':'category'})
print(category_class[1])

<dd class="category tags">
<ul class="commas">
<li><a class="tag" href="/tags/F*s*M/works">F/M</a></li><li><a class="tag" href="/tags/Gen/works">Gen</a></li>
</ul>
</dd>


In [637]:
category_list = []
for item in category_class:
    tag = category_class[1].find('a')
    string = tag.string
    category_list.append(string)
category_list = list(set(category_list))
if len(category_list) == 1:
    category_list = category_list[0]
else:
    pass
print(category_list)

F/M


In [638]:
def get_category1(soup):
    category_class = soup.find_all(attrs={'class':'category'})
    if len(category_class) == 0:
        category_class = soup.find_all(attrs={'class':'category tags'})
        if len(category_class) == 0:
            category = "No categories tagged"
            return category
        else:
            category_class1 = category_class[1]
            tag = category_class1.find('a')
            string = str(tag.string)
            category = string
            return category

    else:
        category_list = []
        for item in category_class:
            tag = category_class[1].find('a')
            string = tag.string
            category_list.append(string)
        category_list = list(set(category_list))
        if len(category_list) == 1:
            category_list = category_list[0]
        else:
            pass
        return category_list





        # category_list = []
        # for item in category_class:
            # tag = item
            # string = tag.string
            # category_list.append(string)
        # rating_list = list(set(category_list))
        # for i in range(len(category_list)):
            # category_list[i] = str(category_list[i])
        # if len(category_list) == 1:
            # category_list = category_list[0]
        # else:
            # pass
    # return category_list

In [620]:
warning_class = soup.find_all(attrs={'class':'warning'})
print(warning_class)

[<dt class="warning tags">
<a href="/tos_faq#tags">Archive Warning</a>:
          </dt>, <dd class="warning tags">
<ul class="commas">
<li><a class="tag" href="/tags/Graphic%20Depictions%20Of%20Violence/works">Graphic Depictions Of Violence</a></li>
</ul>
</dd>]


In [614]:
print(warning_class[1])

<dd class="warning tags">
<ul class="commas">
<li><a class="tag" href="/tags/Graphic%20Depictions%20Of%20Violence/works">Graphic Depictions Of Violence</a></li>
</ul>
</dd>


In [603]:
warning_class1 = warning_class[1]
warning_list = []
for child in warning_class1:
    tag = warning_class1.find('a')
    if tag is not None:
        string = str(tag.string)
        warning_list.append(string)
    else:
        pass

warning_list = list(set(warning_list))
print(warning_list)

['Graphic Depictions Of Violence']


In [552]:
rating_class = soup.find_all(attrs={'class':'rating tags'})

if len(rating_class) == 0:
    rating = "Unknown"
else:
    rating_class1 = rating_class[1]
    tag = rating_class1.find('a')
    string = tag.string
    rating = string
    print(rating)

Teen And Up Audiences


In [473]:
warning_class = soup.find_all(attrs={'class':'warning tags'})
warning_class1 = warning_class[1]
tag = warning_class1.find('a')
string = tag.string
print(string)

Graphic Depictions Of Violence


In [474]:
category_class = soup.find_all(attrs={'class':'category tags'})
category_class1 = category_class[1]
tag = category_class1.find('a')
string = tag.string
print(string)

F/M


In [494]:
fandom_class = soup.find_all(attrs={'class':'fandom tags'})
fandom_class1 = fandom_class[1]
fandom_list = []
for child in fandom_class1:
    tag = fandom_class1.find('a')
    string = tag.string
    fandom_list.append(string)
fandom_list = list(set(fandom_list))
print(fandom_list)

['Candela Obscura (Critical Role Web Series)']


In [521]:
relationship_class = soup.find_all(attrs={'class':'relationship tags'})
relationship_class1 = relationship_class[1]

for child in relationship_class1:
    tag_list = relationship_class1.find_all('a')
    relationship_list = []
    for tag in tag_list:
        string = tag.string
        relationship_list.append(string)

if len(relationship_list) == 1:
    relationship_list = relationship_list[0]

print(relationship_list)

Jinnah "Jean" Basar/Marion Collodi


In [522]:
character_class = soup.find_all(attrs={'class':'character tags'})
character_class1 = character_class[1]

for child in character_class1:
    tag_list = character_class1.find_all('a')
    character_list = []
    for tag in tag_list:
        string = tag.string
        character_list.append(string)

if len(character_list) == 1:
    character_list = character_list[0]
else:
    pass

print(character_list)

['Jinnah "Jean" Basar', 'Marion Collodi', 'Beatrix Monroe | Auntie Bee']


In [525]:
freeform_class = soup.find_all(attrs={'class':'freeform tags'})
freeform_class1 = freeform_class[1]

for child in freeform_class1:
    tag_list = freeform_class1.find_all('a')
    freeform_list = []
    for tag in tag_list:
        string = tag.string
        freeform_list.append(string)

if len(freeform_list) == 1:
    freeform_list = freeform_list[0]
else:
    pass

print(freeform_list)

["Sean is also very present but he's not like. conscious", "there's ot3 energy here but this fic is About marion and jean", 'Pre-Canon', 'Surgery', 'Cuddles', 'get loved idiot']


## First 10 fanworks

In [430]:
first_10_fanworks = fanwork_links[0:10]
print(first_10_fanworks)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606']


In [497]:
## test: first 10 flinks
total_fanworks = len(first_10_fanworks)
first10works_dict = create_all_works_dict()
for i in range(len(first_10_fanworks)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))
    current_work_html = get_fanwork_html(first_10_fanworks[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))
    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))
    current_work_dict = create_fanwork_dict(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))
    append_all_works_dict(one_fanwork_dict, first10works_dict)
    print("Successfully appended dictionary to first10works_dict for link: ", str(i+1), " of ", str(total_fanworks))

Current fanwork link:  1 of 10
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  10
Successfully created soup for link:  1  of  10
Successfully created dictionary for link:  1  of  10
Successfully appended dictionary to first10works_dict for link:  1  of  10
Current fanwork link:  2 of 10
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  10
Successfully created soup for link:  2  of  10


UnboundLocalError: cannot access local variable 'warnings_list' where it is not associated with a value

In [498]:
print(first10works_dict)

{1: {'Rating': 'Explicit', 'Archive Warnings': 'No Archive Warnings Apply', 'Category': 'M/M', 'Fandom': 'Candela Obscura (Critical Role Web Series)', 'Characters': ['Leo Amicus', 'Original Male Character(s)'], 'Relationships': 'Leo Amicus/Original Character(s)', 'Additional Tags': ['Circle of the Crimson Mirror (Candela Obscura)', 'Boats and Ships', 'Crew of the Dandridge', 'Missing Scene', 'Blow Jobs', 'Anal Sex', 'Topping from the Bottom', 'Plot What Plot/Porn Without Plot', 'almost', 'Age Difference'], 'Language': 'English', 'Date Published': 'Unknown', 'Date Updated': ['09 Nov 2025'], 'Word Count': 1070, 'Chapter Count': '1/2', 'Comment Count': 2, 'Kudos Count': 2, 'Bookmarks Count': 0}}


In [ ]:
print(len(first10works_dict))
print(first10works_dict)

In [ ]:
# convert to json and save to file
first10works_json = convert_to_json(first10works_dict)
filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/first10works_test.json'
save_json_to_file(filepath, first10works_json)

## First 20 fanworks - for G8

In [ ]:
first_20_fanworks = fanwork_links[0:20]
print(first_20_fanworks)

In [ ]:
## first20works_dict
total_fanworks = len(first_20_fanworks)
first20works_dict = {}
for i in range(len(first_20_fanworks)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(first_20_fanworks[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    # print(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    all_dd_stripped = strip_dd(all_dd)
    # print(len(all_dd_stripped))
    all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
    all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
    all_dd_stripped = cleanup_character_tags(all_dd_stripped)
    all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
    print("Successfully cleaned all_dd_stripped for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
    # print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, first20works_dict)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to first20works_dict.")

In [ ]:
## convert to json and save to file
first20works_json = convert_to_json(first20works_dict)
with open('./data/first20works.json', "w") as f:
    f.write(first20works_json)

## Full fanworks list

In [ ]:
print(len(fanwork_links))

In [ ]:
fanwork_links_minus1 = fanwork_links[1:]

In [ ]:
print(len(fanwork_links_minus1))

In [ ]:
## full fanworks list
total_fanworks = len(fanwork_links)
all_candela_works_dict = {}
for i in range(len(fanwork_links)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(fanwork_links[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    # print(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    all_dd_stripped = strip_dd(all_dd)
    # print(len(all_dd_stripped))
    all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
    all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
    all_dd_stripped = cleanup_character_tags(all_dd_stripped)
    all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
    print("Successfully cleaned all_dd_stripped for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
    # print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_candela_works_dict)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_candela_works_dict.")

In [ ]:
print(len(all_candela_works_dict))

In [ ]:
## convert to json and save to file
all_candela_works_json = convert_to_json(all_candela_works_dict)
with open('../../data/allcandelaworks.json', "w") as f:
    f.write(all_candela_works_json)

## Full fanworks - from individual pagination links

In [ ]:
def get_links_to_scrape(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    links_to_scrape = open(filepath).readlines()
    return links_to_scrape

In [ ]:
# test get_links_to_scrape
links_to_scrape = get_links_to_scrape('./data/txt_files.txt')
print(links_to_scrape)

In [ ]:
## get pagination links to scrape
pagination_links1 = get_fanwork_links('./data/txt_files/fanwork_links1.txt')
pagination_links2 = get_fanwork_links('./data/txt_files/fanwork_links2.txt')
pagination_links3 = get_fanwork_links('./data/txt_files/fanwork_links3.txt')
pagination_links4 = get_fanwork_links('./data/txt_files/fanwork_links4.txt')
pagination_links5 = get_fanwork_links('./data/txt_files/fanwork_links5.txt')
pagination_links6 = get_fanwork_links('./data/txt_files/fanwork_links6.txt')
pagination_links7 = get_fanwork_links('./data/txt_files/fanwork_links7.txt')
pagination_links8 = get_fanwork_links('./data/txt_files/fanwork_links8.txt')
pagination_links9 = get_fanwork_links('./data/txt_files/fanwork_links9.txt')
pagination_links10 = get_fanwork_links('./data/txt_files/fanwork_links10.txt')
pagination_links11 = get_fanwork_links('./data/txt_files/fanwork_links11.txt')

In [ ]:
print(pagination_links1)
print(len(pagination_links1))

In [ ]:
pagination_links = [pagination_links1, pagination_links2, pagination_links3, pagination_links4, pagination_links5, pagination_links6, pagination_links7, pagination_links8, pagination_links9, pagination_links10, pagination_links11]

In [ ]:
print(len(pagination_links))
print(pagination_links)

In [ ]:
def remove_chapter_links(work_links):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only_final = work_links
    for i in range(len(work_links)):
        current_link = work_links[i]
        if 'chapters' in current_link:
            works_only_final.remove(current_link)
        else:
            continue
    return works_only_final

In [ ]:
# remove chapters links from each pagination link
for i in range(len(pagination_links1)):
    current_link = pagination_links1[i]
    if 'chapters' in current_link:
        pagination_links1.remove(current_link)
    else:
        continue

In [ ]:
print(pagination_links1)
print(len(pagination_links1))

In [ ]:
print(len(pagination_links))

In [ ]:
## individual pagination links list

all_candela_works_dict1 = {}

for i in range(len(pagination_links)):
    total_fanworks = len(pagination_links[i])
    current_dict = {}
    current_pagination_link = pagination_links[i]
    print("Current pagination link: " + str(i+1) + " of", str(len(pagination_links)))
    for index in range(len(current_pagination_link)):
        print("Current fanwork link: " + str(index+1), " of", total_fanworks)

        ## load fanworks and create soup objects
        soup = get_html_doc_fanworks(fanwork_links[index])
        all_dt = get_all_dt(soup)
        all_dt_updated = update_all_dt(all_dt)
        # print(all_dt_updated)
        all_dd = get_all_dd(soup)
        print("Successfully loaded fanworks and created soup objects for link", str(index+1))

        ## clean up all_dd
        all_dd_stripped = strip_dd(all_dd)
        # print(len(all_dd_stripped))
        all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)
        all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)
        all_dd_stripped = cleanup_character_tags(all_dd_stripped)
        all_dd_stripped = cleanup_additional_tags(all_dd_stripped)
        print("Successfully cleaned all_dd_stripped for link", str(index+1))

        ## create individual fanwork dictionary
        fanwork_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
        # print(fanwork_dict)
        print("Successfully created individual fanwork dictionary for link", str(index+1))

        ## append fanwork dictionary to current dictionary
        append_all_works_dict(fanwork_dict, current_dict)
        print("Successfully appended individual fanwork dictionary", str(index+1), "to current dict.")

    ## convert current dict to json and save to file
    current_json = convert_to_json(current_dict)
    json_filepath = './data/json_files/candela_page' + str(i+1) + '.json'
    with open(json_filepath, "w") as f:
        f.write(current_json)

# what am I trying to do here
# I'm trying to save each json file with a new name so they don't overwrite

In [ ]:
# code from: https://www.geeksforgeeks.org/python/how-to-merge-multiple-json-files-using-python/
def merge_json_files(file_paths, output_file):
    merged_data = []
    for path in file_paths:
        with open(path, 'r') as file:
            data = json.load(file)
            merged_data.append(data)
    with open(output_file, 'w') as outfile:
        json.dump(merged_data, outfile)

In [ ]:
# append individual jsons to all_candela_works_dict1
file_paths = ["data/json_files/candela_page1.json",
              "data/json_files/candela_page2.json",
              "data/json_files/candela_page3.json",
              "data/json_files/candela_page4.json",
              "data/json_files/candela_page5.json",
              "data/json_files/candela_page6.json",
              "data/json_files/candela_page7.json",
              "data/json_files/candela_page8.json",
              "data/json_files/candela_page9.json",
              "data/json_files/candela_page10.json",
              "data/json_files/candela_page11.json",
              ]

# create new JSON file
output_dict = {}
output_file = convert_to_json(output_dict)
output_filepath = "./data/json_files/candela_merged.json"
with open(output_filepath, 'w') as f:
    f.write(output_file)

merge_json_files(file_paths, output_filepath)
print(f"Merged data written to '{output_file}'")

In [ ]:
print(len(all_candela_works_dict1))

In [ ]:
## convert to json and save to file
all_candela_works_json1 = convert_to_json(all_candela_works_dict1)
with open('./data/allcandelaworks.json', "w") as f:
    f.write(all_candela_works_json1)

# Testing / Troubleshooting

## Round 1

In [ ]:
def check_if_rating(all_dt):
    if 'Rating' in all_dt_updated:
        print('Rating is in all_dt_updated')
    else:
        print("nope")

In [ ]:
for i in range(len(fanwork_links)):
    fanwork_links[i] = fanwork_links[i].replace("\n","")

In [ ]:
test_fanwork_list = fanwork_links[0:10]
print(test_fanwork_list)

In [ ]:
print(test_fanwork_list[0])

In [ ]:
def check_word_count(all_dt):
    if 'Word Count' in all_dt:
        print('Word Count is in all_dt')
    else:
        print("Word count is not in all_dt")

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list)
all_works_dict2 = {}
for i in range(len(test_fanwork_list)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    check_word_count(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_works_dict2)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict2.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json = convert_to_json(all_works_dict2)

# save json to file
with open('../../data/test_candela_works.json', "w") as f:
    f.write(test_candela_works_json)

## Round 2

In [ ]:
test_fanwork_list1 = fanwork_links[0:1]
print(test_fanwork_list1)
print(type(test_fanwork_list1))

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list1)
all_works_dict3 = {}
for i in range(len(test_fanwork_list1)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list1[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    print(all_dt_updated[2])
    print(type(all_dt_updated[2]))
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict3
    append_all_works_dict(fanwork_dict, all_works_dict3)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict3.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json1 = convert_to_json(all_works_dict3)

# save json to file
with open('../../data/test_candela_works1.json', "w") as f:
    f.write(test_candela_works_json1)

## Round 3

In [ ]:
test_fanwork_list2 = fanwork_links[1:2]
print(test_fanwork_list2)

In [ ]:
a = [10, 20, 30, 40, 50]

if 30 in a:
    print("Element exists in the list")
else:
    print("Element does not exist")

In [ ]:
# create + fill all_works_dict4
total_fanworks = len(test_fanwork_list2)
all_works_dict4 = {}
for i in range(len(test_fanwork_list2)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list2[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict4
    append_all_works_dict(fanwork_dict, all_works_dict4)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict4.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json2 = convert_to_json(all_works_dict4)

# save json to file
with open('../../data/test_candela_works2.json', "w") as f:
    f.write(test_candela_works_json2)